# 01 — Data Profiling & EDA (Lending Club)

First stage of the credit-risk pipeline: load the raw Lending Club accepted-loans file
(2007–2018Q4), understand its structure, and select the application-time columns that
all later stages build on.

**What this notebook decides**

* From 151 raw columns, keep the 38 that are known at application time. Post-origination fields (`out_prncp`, `total_pymnt`, `recoveries`, `last_fico_*`, …) are excluded — they would leak the loan outcome into the model.
* Serverless Spark cannot write to `/Workspace` paths, so the selection is persisted as a Unity Catalog table — `workspace.default.lending_club` — the stable hand-off point for the cleaning notebook.
* Coverage of the three bureau columns added late to the schema (`tot_cur_bal`, `tot_coll_amt`, `collections_12_mths_ex_med`) is verified per vintage window before they are used as features.

In [1]:
#Load the data

data = '/Workspace/Users/<your_username>/Credit_Risk_Model(Lending_Club)/accepted_2007_to_2018Q4.csv.gz'

df_lending_club = spark.read.csv(data, header=True, inferSchema=True)

In [2]:
df_lending_club.printSchema()

root
 |-- id: string (nullable = true)
 |-- member_id: string (nullable = true)
 |-- loan_amnt: double (nullable = true)
 |-- funded_amnt: double (nullable = true)
 |-- funded_amnt_inv: double (nullable = true)
 |-- term: string (nullable = true)
 |-- int_rate: double (nullable = true)
 |-- installment: double (nullable = true)
 |-- grade: string (nullable = true)
 |-- sub_grade: string (nullable = true)
 |-- emp_title: string (nullable = true)
 |-- emp_length: string (nullable = true)
 |-- home_ownership: string (nullable = true)
 |-- annual_inc: string (nullable = true)
 |-- verification_status: string (nullable = true)
 |-- issue_d: string (nullable = true)
 |-- loan_status: string (nullable = true)
 |-- pymnt_plan: string (nullable = true)
 |-- url: string (nullable = true)
 |-- desc: string (nullable = true)
 |-- purpose: string (nullable = true)
 |-- title: string (nullable = true)
 |-- zip_code: string (nullable = true)
 |-- addr_state: string (nullable = true)
 |-- dti: string 

In [3]:
df_lending_club.show()

+--------+---------+---------+-----------+---------------+----------+--------+-----------+-----+---------+--------------------+----------+--------------+----------+-------------------+--------+-----------+----------+--------------------+----+------------------+--------------------+--------+----------+-----+-----------+----------------+--------------+---------------+--------------+----------------------+----------------------+--------+-------+---------+----------+---------+-------------------+---------+-------------+------------------+---------------+---------------+-------------+------------------+----------+-----------------------+------------+---------------+------------+------------------+--------------------+-------------------+--------------------------+---------------------------+-----------+----------------+----------------+---------+-------------------------+--------------+------------+-----------+-----------+-----------+-----------+-----------+------------------+------------+-

In [4]:
#print the shape of the data

print(f'rows: {df_lending_club.count()}', f'columns: {len(df_lending_club.columns)}')

rows: 2260701 columns: 151


## Column selection and persistence

Keep the 38 application-time columns — borrower attributes, loan terms, LC grades and
the bureau snapshot available when the application is scored. Outcome columns are
deliberately excluded (leakage prevention). The selection is written once to a Unity
Catalog table and read back to verify.

In [5]:
#select the columns we want to keep

df_lending_club = df_lending_club.select('loan_amnt', 'funded_amnt', 'funded_amnt_inv', 'term', 'int_rate', 'installment', 'grade', 'sub_grade', 'emp_length', 'home_ownership', 'annual_inc', 'verification_status', 'issue_d', 'loan_status', 'purpose', 'addr_state', 'dti', 'delinq_2yrs', 'earliest_cr_line', 'fico_range_low', 'fico_range_high', 'inq_last_6mths', 'mths_since_last_delinq', 'mths_since_last_record', 'open_acc', 'pub_rec', 'revol_bal', 'revol_util', 'total_acc', 'initial_list_status', 'application_type', 'acc_now_delinq', 'pub_rec_bankruptcies', 'tax_liens', 'mort_acc', 'tot_cur_bal', 'tot_coll_amt', 'collections_12_mths_ex_med')

#display the data

display(df_lending_club)
#write the data (serverless Spark cannot write data to /Workspace paths, so persist to a Unity Catalog table instead)

df_lending_club.write.mode('overwrite').option('overwriteSchema', 'true').saveAsTable('workspace.default.lending_club')
#read the data back from the table

df_lending_club = spark.read.table('workspace.default.lending_club')
#display the data

display(df_lending_club)

loan_amnt,funded_amnt,funded_amnt_inv,term,int_rate,installment,grade,sub_grade,emp_length,home_ownership,annual_inc,verification_status,issue_d,loan_status,purpose,addr_state,dti,delinq_2yrs,earliest_cr_line,fico_range_low,fico_range_high,inq_last_6mths,mths_since_last_delinq,mths_since_last_record,open_acc,pub_rec,revol_bal,revol_util,total_acc,initial_list_status,application_type,acc_now_delinq,pub_rec_bankruptcies,tax_liens,mort_acc,tot_cur_bal,tot_coll_amt,collections_12_mths_ex_med
3600.0,3600.0,3600.0,36 months,13.99,123.03,C,C4,10+ years,MORTGAGE,55000.0,Not Verified,Dec-2015,Fully Paid,debt_consolidation,PA,5.91,0.0,Aug-2003,675.0,679.0,1.0,30.0,,7.0,0.0,2765.0,29.7,13.0,w,Individual,0.0,0.0,0.0,1.0,144904.0,722.0,0.0
24700.0,24700.0,24700.0,36 months,11.99,820.28,C,C1,10+ years,MORTGAGE,65000.0,Not Verified,Dec-2015,Fully Paid,small_business,SD,16.06,1.0,Dec-1999,715.0,719.0,4.0,6.0,,22.0,0.0,21470.0,19.2,38.0,w,Individual,0.0,0.0,0.0,4.0,204396.0,0.0,0.0
20000.0,20000.0,20000.0,60 months,10.78,432.66,B,B4,10+ years,MORTGAGE,63000.0,Not Verified,Dec-2015,Fully Paid,home_improvement,IL,10.78,0.0,Aug-2000,695.0,699.0,0.0,,,6.0,0.0,7869.0,56.2,18.0,w,Joint App,0.0,0.0,0.0,5.0,189699.0,0.0,0.0
35000.0,35000.0,35000.0,60 months,14.85,829.9,C,C5,10+ years,MORTGAGE,110000.0,Source Verified,Dec-2015,Current,debt_consolidation,NJ,17.06,0.0,Sep-2008,785.0,789.0,0.0,,,13.0,0.0,7802.0,11.6,17.0,w,Individual,0.0,0.0,0.0,1.0,301500.0,0.0,0.0
10400.0,10400.0,10400.0,60 months,22.45,289.91,F,F1,3 years,MORTGAGE,104433.0,Source Verified,Dec-2015,Fully Paid,major_purchase,PA,25.37,1.0,Jun-1998,695.0,699.0,3.0,12.0,,12.0,0.0,21929.0,64.5,35.0,w,Individual,0.0,0.0,0.0,6.0,331730.0,0.0,0.0
11950.0,11950.0,11950.0,36 months,13.44,405.18,C,C3,4 years,RENT,34000.0,Source Verified,Dec-2015,Fully Paid,debt_consolidation,GA,10.2,0.0,Oct-1987,690.0,694.0,0.0,,,5.0,0.0,8822.0,68.4,6.0,w,Individual,0.0,0.0,0.0,0.0,12798.0,0.0,0.0
20000.0,20000.0,20000.0,36 months,9.17,637.58,B,B2,10+ years,MORTGAGE,180000.0,Not Verified,Dec-2015,Fully Paid,debt_consolidation,MN,14.67,0.0,Jun-1990,680.0,684.0,0.0,49.0,,12.0,0.0,87329.0,84.5,27.0,f,Individual,0.0,0.0,0.0,4.0,360358.0,0.0,0.0
20000.0,20000.0,20000.0,36 months,8.49,631.26,B,B1,10+ years,MORTGAGE,85000.0,Not Verified,Dec-2015,Fully Paid,major_purchase,SC,17.61,1.0,Feb-1999,705.0,709.0,0.0,3.0,,8.0,0.0,826.0,5.7,15.0,w,Individual,0.0,0.0,0.0,3.0,141601.0,0.0,0.0
10000.0,10000.0,10000.0,36 months,6.49,306.45,A,A2,6 years,RENT,85000.0,Not Verified,Dec-2015,Fully Paid,credit_card,PA,13.07,0.0,Apr-2002,685.0,689.0,1.0,,106.0,14.0,1.0,10464.0,34.5,23.0,w,Individual,0.0,1.0,0.0,1.0,27957.0,8341.0,0.0
8000.0,8000.0,8000.0,36 months,11.48,263.74,B,B5,10+ years,MORTGAGE,42000.0,Not Verified,Dec-2015,Fully Paid,credit_card,RI,34.8,0.0,Nov-1994,700.0,704.0,0.0,75.0,,8.0,0.0,7034.0,39.1,18.0,w,Individual,0.0,0.0,0.0,1.0,199696.0,0.0,0.0


loan_amnt,funded_amnt,funded_amnt_inv,term,int_rate,installment,grade,sub_grade,emp_length,home_ownership,annual_inc,verification_status,issue_d,loan_status,purpose,addr_state,dti,delinq_2yrs,earliest_cr_line,fico_range_low,fico_range_high,inq_last_6mths,mths_since_last_delinq,mths_since_last_record,open_acc,pub_rec,revol_bal,revol_util,total_acc,initial_list_status,application_type,acc_now_delinq,pub_rec_bankruptcies,tax_liens,mort_acc,tot_cur_bal,tot_coll_amt,collections_12_mths_ex_med
3600.0,3600.0,3600.0,36 months,13.99,123.03,C,C4,10+ years,MORTGAGE,55000.0,Not Verified,Dec-2015,Fully Paid,debt_consolidation,PA,5.91,0.0,Aug-2003,675.0,679.0,1.0,30.0,,7.0,0.0,2765.0,29.7,13.0,w,Individual,0.0,0.0,0.0,1.0,144904.0,722.0,0.0
24700.0,24700.0,24700.0,36 months,11.99,820.28,C,C1,10+ years,MORTGAGE,65000.0,Not Verified,Dec-2015,Fully Paid,small_business,SD,16.06,1.0,Dec-1999,715.0,719.0,4.0,6.0,,22.0,0.0,21470.0,19.2,38.0,w,Individual,0.0,0.0,0.0,4.0,204396.0,0.0,0.0
20000.0,20000.0,20000.0,60 months,10.78,432.66,B,B4,10+ years,MORTGAGE,63000.0,Not Verified,Dec-2015,Fully Paid,home_improvement,IL,10.78,0.0,Aug-2000,695.0,699.0,0.0,,,6.0,0.0,7869.0,56.2,18.0,w,Joint App,0.0,0.0,0.0,5.0,189699.0,0.0,0.0
35000.0,35000.0,35000.0,60 months,14.85,829.9,C,C5,10+ years,MORTGAGE,110000.0,Source Verified,Dec-2015,Current,debt_consolidation,NJ,17.06,0.0,Sep-2008,785.0,789.0,0.0,,,13.0,0.0,7802.0,11.6,17.0,w,Individual,0.0,0.0,0.0,1.0,301500.0,0.0,0.0
10400.0,10400.0,10400.0,60 months,22.45,289.91,F,F1,3 years,MORTGAGE,104433.0,Source Verified,Dec-2015,Fully Paid,major_purchase,PA,25.37,1.0,Jun-1998,695.0,699.0,3.0,12.0,,12.0,0.0,21929.0,64.5,35.0,w,Individual,0.0,0.0,0.0,6.0,331730.0,0.0,0.0
11950.0,11950.0,11950.0,36 months,13.44,405.18,C,C3,4 years,RENT,34000.0,Source Verified,Dec-2015,Fully Paid,debt_consolidation,GA,10.2,0.0,Oct-1987,690.0,694.0,0.0,,,5.0,0.0,8822.0,68.4,6.0,w,Individual,0.0,0.0,0.0,0.0,12798.0,0.0,0.0
20000.0,20000.0,20000.0,36 months,9.17,637.58,B,B2,10+ years,MORTGAGE,180000.0,Not Verified,Dec-2015,Fully Paid,debt_consolidation,MN,14.67,0.0,Jun-1990,680.0,684.0,0.0,49.0,,12.0,0.0,87329.0,84.5,27.0,f,Individual,0.0,0.0,0.0,4.0,360358.0,0.0,0.0
20000.0,20000.0,20000.0,36 months,8.49,631.26,B,B1,10+ years,MORTGAGE,85000.0,Not Verified,Dec-2015,Fully Paid,major_purchase,SC,17.61,1.0,Feb-1999,705.0,709.0,0.0,3.0,,8.0,0.0,826.0,5.7,15.0,w,Individual,0.0,0.0,0.0,3.0,141601.0,0.0,0.0
10000.0,10000.0,10000.0,36 months,6.49,306.45,A,A2,6 years,RENT,85000.0,Not Verified,Dec-2015,Fully Paid,credit_card,PA,13.07,0.0,Apr-2002,685.0,689.0,1.0,,106.0,14.0,1.0,10464.0,34.5,23.0,w,Individual,0.0,1.0,0.0,1.0,27957.0,8341.0,0.0
8000.0,8000.0,8000.0,36 months,11.48,263.74,B,B5,10+ years,MORTGAGE,42000.0,Not Verified,Dec-2015,Fully Paid,credit_card,RI,34.8,0.0,Nov-1994,700.0,704.0,0.0,75.0,,8.0,0.0,7034.0,39.1,18.0,w,Individual,0.0,0.0,0.0,1.0,199696.0,0.0,0.0


## Coverage of the late-added bureau columns

`tot_cur_bal` / `tot_coll_amt` / `collections_12_mths_ex_med` were only reported by
Lending Club from ~2012. Before using them as features, confirm their null coverage in
the modeling windows (train 2012–2015 vs test 2016–2018) and check for malformed
values (e.g. rows where the CSV columns shifted).

In [6]:
# Null coverage of the newly added bureau columns, split by the same vintage windows
# used in modeling (train = 2012-2015, test = 2016-2018). These fields only started being
# reported by Lending Club around 2012, so coverage in the train window must be confirmed
# before using them as features.

from pyspark.sql import functions as F

new_bureau_cols = ['tot_cur_bal', 'tot_coll_amt', 'collections_12_mths_ex_med']

nulls_by_period = (
    df_lending_club
    .withColumn('year', F.substring('issue_d', 5, 4))
    .filter(F.col('year').isNotNull() & F.col('year').between('2012', '2018'))
    .withColumn('period', F.when(F.col('year') <= '2015', 'train_2012_2015').otherwise('test_2016_2018'))
    .groupBy('period')
    .agg(F.count('*').alias('n'), *[F.round(F.avg(F.col(c).isNull().cast('double')), 4).alias(c + '_null_rate') for c in new_bureau_cols])
)

display(nulls_by_period)

# Also check data quality of the new columns in detail: nulls, malformed (unparseable)
# values, and non-zero rate for collections_12_mths_ex_med (it is usually 0).
# try_cast returns NULL for values that cannot be parsed as numbers (e.g. shifted
# rows like 'Jan-2018' that ended up in these columns).

quality = (
    df_lending_club
    .withColumn('year', F.substring('issue_d', 5, 4))
    .filter(F.col('year').isNotNull() & F.col('year').between('2012', '2018'))
    .withColumn('period', F.when(F.col('year') <= '2015', 'train_2012_2015').otherwise('test_2016_2018'))
    .groupBy('period')
    .agg(
        F.count('*').alias('n'),
        *[F.round(F.avg(F.col(c).isNull().cast('double')), 4).alias(c + '_null_rate') for c in new_bureau_cols],
        *[F.round(F.avg((F.col(c).isNotNull() & F.expr(f"try_cast({c} AS DOUBLE)").isNull()).cast('double')), 4).alias(c + '_malformed_rate') for c in new_bureau_cols],
    )
)
display(quality)

# Non-zero rate of collections_12_mths_ex_med among well-formed values
zeros = (
    df_lending_club
    .withColumn('parsed', F.expr("try_cast(collections_12_mths_ex_med AS DOUBLE)"))
    .filter(F.col('parsed').isNotNull())
    .agg(F.round(F.avg((F.col('parsed') > 0).cast('double')), 4).alias('pct_nonzero'))
)
display(zeros)

period,n,tot_cur_bal_null_rate,tot_coll_amt_null_rate,collections_12_mths_ex_med_null_rate
train_2012_2015,844904,0.0328,0.0328,0.0
test_2016_2018,1373228,0.0,0.0,0.0


period,n,tot_cur_bal_null_rate,tot_coll_amt_null_rate,collections_12_mths_ex_med_null_rate,tot_cur_bal_malformed_rate,tot_coll_amt_malformed_rate,collections_12_mths_ex_med_malformed_rate
train_2012_2015,844904,0.0328,0.0328,0.0,0.0,0.0,0.0
test_2016_2018,1373228,0.0,0.0,0.0,0.0,0.0,0.0


pct_nonzero
0.0166


## Profiling findings

* Raw file: **2,260,701 accepted loans × 151 columns** (2007–2018Q4).
* 38 application-time columns selected; post-origination outcome columns excluded (leakage prevention).
* Bureau columns: ~3.3% null in the 2012–2015 window, fully populated 2016–2018, zero malformed values — usable with missing-indicator flags (added in the cleaning notebook).
* Output of this notebook: `workspace.default.lending_club` (2,260,701 × 38), the input for the `02_data_cleaning` notebook.